# LayoutLM Fine-Tuning — SROIE Receipt NER

## Section 1 — Installation

In [ ]:
!pip install -q transformers datasets "seqeval==1.2.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


## Section 2 — Upload Dataset

Run `python scripts/prepare_dataset.py` locally first.
Then upload `train.json` and `test.json` from `data/processed/layoutlm_dataset/`.

In [ ]:
import os
from google.colab import files

os.makedirs('data/processed/layoutlm_dataset', exist_ok=True)

print('Upload train.json:')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open('data/processed/layoutlm_dataset/train.json', 'wb') as f:
        f.write(content)
    print('train.json uploaded ({:.1f} MB)'.format(len(content) / 1e6))

print('\nUpload test.json:')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open('data/processed/layoutlm_dataset/test.json', 'wb') as f:
        f.write(content)
    print('test.json uploaded ({:.1f} MB)'.format(len(content) / 1e6))

Upload train.json:


## Section 3 — Constants and Model Loading

In [ ]:
import json
import os
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    LayoutLMConfig,
    LayoutLMForTokenClassification,
    LayoutLMTokenizerFast,
)

HF_MODEL_NAME = 'microsoft/layoutlm-base-uncased'
DATASET_DIR = 'data/processed/layoutlm_dataset'
OUTPUT_DIR = 'data/layoutlm_finetuned'
MAX_SEQ_LENGTH = 512

LABEL2ID = {
    'O': 0,
    'B-COMPANY': 1, 'I-COMPANY': 2,
    'B-DATE': 3,    'I-DATE': 4,
    'B-ADDRESS': 5, 'I-ADDRESS': 6,
    'B-TOTAL': 7,   'I-TOTAL': 8,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('Device:', device)
print('Number of labels:', len(LABEL2ID))

In [ ]:
def load_pretrained_flexible(model_name_or_path):
    """
    Load LayoutLM pretrained weights.
    Supports both old format (bert.*) and new format (layoutlm.*).
    Classifier head is not in the pretrained model — starts with random init.
    """
    config = LayoutLMConfig.from_pretrained(model_name_or_path)
    config.num_labels = len(LABEL2ID)
    config.id2label = ID2LABEL
    config.label2id = LABEL2ID

    model = LayoutLMForTokenClassification(config)

    from transformers.modeling_utils import load_state_dict
    from huggingface_hub import hf_hub_download
    try:
        ckpt_path = hf_hub_download(repo_id=model_name_or_path, filename='pytorch_model.bin')
    except Exception:
        ckpt_path = model_name_or_path

    raw_sd = torch.load(ckpt_path, map_location='cpu', weights_only=True)

    first_key = next(iter(raw_sd))
    if first_key.startswith('bert.'):
        print('Old format (bert.*) detected, remapping to layoutlm.*...')
        remapped = {}
        for k, v in raw_sd.items():
            if k.startswith('bert.'):
                remapped['layoutlm.' + k[len('bert.'):]] = v
        raw_sd = remapped
    else:
        print('New format (layoutlm.*) detected, loading directly...')
        raw_sd = {k: v for k, v in raw_sd.items() if not k.startswith('classifier')}

    missing, unexpected = model.load_state_dict(raw_sd, strict=False)
    classifier_missing = [k for k in missing if k.startswith('classifier')]
    other_missing = [k for k in missing if not k.startswith('classifier')]

    print('{} weights loaded.'.format(len(raw_sd)))
    print('Classifier head (expected): {} missing'.format(len(classifier_missing)))
    if other_missing:
        print('WARNING — Unexpected missing keys:', other_missing)
    if unexpected:
        print('WARNING — Unexpected keys:', unexpected)

    return model

print('load_pretrained_flexible defined.')

## Section 4 — Dataset and Tokenizer

In [ ]:
def load_examples(split):
    path = os.path.join(DATASET_DIR, '{}.json'.format(split))
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def tokenize_and_align(tokenizer, example):
    encoding = tokenizer(
        example['words'],
        is_split_into_words=True,
        padding='max_length',
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    word_ids = encoding.word_ids()
    aligned_labels = []
    aligned_boxes = []
    prev_word_id = None
    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)
            aligned_boxes.append([0, 0, 0, 0])
        elif word_id != prev_word_id:
            aligned_labels.append(example['labels'][word_id])
            aligned_boxes.append(example['boxes'][word_id])
        else:
            aligned_labels.append(-100)
            aligned_boxes.append(example['boxes'][word_id])
        prev_word_id = word_id
    encoding['labels'] = aligned_labels
    encoding['bbox'] = aligned_boxes
    return encoding


class SROIEDataset(Dataset):
    def __init__(self, examples, tokenizer):
        self.encodings = [tokenize_and_align(tokenizer, ex) for ex in examples]

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        item = self.encodings[idx]
        return {
            'input_ids':      torch.tensor(item['input_ids'],      dtype=torch.long),
            'attention_mask': torch.tensor(item['attention_mask'], dtype=torch.long),
            'token_type_ids': torch.tensor(item['token_type_ids'], dtype=torch.long),
            'bbox':           torch.tensor(item['bbox'],           dtype=torch.long),
            'labels':         torch.tensor(item['labels'],         dtype=torch.long),
        }


print('Loading tokenizer...')
tokenizer = LayoutLMTokenizerFast.from_pretrained(HF_MODEL_NAME)

print('Loading train examples...')
train_examples = load_examples('train')
print('Loading test examples (validation)...')
val_examples = load_examples('test')
print('Train: {}  Val: {}'.format(len(train_examples), len(val_examples)))

print('Tokenizing...')
train_dataset = SROIEDataset(train_examples, tokenizer)
val_dataset = SROIEDataset(val_examples, tokenizer)
print('Dataset ready.')

## Section 5 — Training

In [ ]:
EPOCHS = 10
BATCH_SIZE = 16
LR = 5e-5

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print('Loading model: {} ...'.format(HF_MODEL_NAME))
model = load_pretrained_flexible(HF_MODEL_NAME)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LR)

print('\nStarting training ({} epochs, batch={}, lr={})...\n'.format(EPOCHS, BATCH_SIZE, LR))

best_val_loss = float('inf')
os.makedirs(OUTPUT_DIR, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    total_train_loss = 0.0
    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        bbox           = batch['bbox'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                        token_type_ids=token_type_ids, bbox=bbox, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    avg_train = total_train_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            bbox           = batch['bbox'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids, bbox=bbox, labels=labels)
            total_val_loss += outputs.loss.item()
    avg_val = total_val_loss / len(val_loader)

    marker = '  <-- best' if avg_val < best_val_loss else ''
    print('Epoch {:>2}/{}: train={:.4f}  val={:.4f}{}'.format(
        epoch, EPOCHS, avg_train, avg_val, marker))

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

print('\nTraining complete. Best val_loss: {:.4f}'.format(best_val_loss))
print('Model saved to: {}'.format(OUTPUT_DIR))

## Section 6 — Download Model

In [ ]:
import shutil
from google.colab import files

zip_path = 'layoutlm_finetuned'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print('layoutlm_finetuned.zip created, starting download...')
files.download(zip_path + '.zip')